# OmeZarrWriterPlate on the Ti2-E

Same experiment as `ome_zarr_writer.ipynb`, stored with
`OmeZarrWriterPlate`: every position becomes a well of a one-row
plate, so napari-ome-zarr tiles the positions side by side instead of
showing them on a slider.

## 1. Connect to the Ti2-E

`Ti2E()` loads `NikonTi2e.cfg`, applies the `System/Startup` preset and
selects the `Channel` group. Pass `config=r"..."` to load another file.
Close NIS-Elements and any other Micro-Manager first: a device can only be
opened by one program at a time.

In [ ]:
from faro_aoki.ti2e import Ti2E
import faro.core.utils as utils

mic = Ti2E()
utils.print_configs(mic.mmc)

## 2. Settings

Two imaging channels on every frame, blue light through the `GFP` preset on
the stim frames, and a reference channel on the last frame. The positions
start at the current stage position and step `STEP_UM` in x, so the sample
must reach that far. Positions have no `z`, so the focus is not moved.

In [ ]:
import os
import tempfile
from datetime import datetime

IMAGING_CHANNELS = [
    {"config": "iRFP", "exposure": 100},  # nuclear marker, segmented
    {"config": "RFP", "exposure": 100},
]
STIM_CHANNEL = {"config": "GFP", "exposure": 100}  # blue light, whole FOV
REF_CHANNEL = {"config": "YFP_Custom", "exposure": 200}  # last frame only
N_POSITIONS = 3
STEP_UM = 300.0  # x distance between positions
INTERVAL_S = 5.0
N_FRAMES = 6
STIM_FRAMES = [1, 3]

path = os.path.join(
    tempfile.gettempdir(), f"faro-ti2e-omezarr-plate-{datetime.now():%Y%m%d-%H%M%S}"
)
print("results go to", path)

## 3. Set up the pipeline

Otsu segmentation on the first imaging channel, trackpy, `SimpleFE`, and
`StimWholeFOV`: the Ti2-E has no DMD, so stimulation lights the whole
field of view.

In [ ]:
from faro.core.data_structures import SegmentationMethod
from faro.core.pipeline import ImageProcessingPipeline
from faro.feature_extraction.simple import SimpleFE
from faro.segmentation.base import OtsuSegmentator
from faro.stimulation.base import StimWholeFOV
from faro.tracking.trackpy import TrackerTrackpy

segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=OtsuSegmentator(),
        use_channel=0,
        save_tracked=True,
    )
]

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=SimpleFE("labels"),
    tracker=TrackerTrackpy(),
    stimulator=StimWholeFOV(),
)

## 4. Create the OmeZarrWriterPlate

The only difference from `ome_zarr_writer.ipynb`.

In [ ]:
from faro.core.writers import OmeZarrWriterPlate

writer = OmeZarrWriterPlate(storage_path=path, store_stim_images=True)
print(f"Zarr store: {writer._zarr_path}")

## 5. Define the experiment

In [ ]:
from faro.core.data_structures import RTMSequence

x0, y0 = mic.mmc.getXYPosition()
positions = [{"x": x0 + i * STEP_UM, "y": y0} for i in range(N_POSITIONS)]

seq = RTMSequence(
    time_plan={"interval": INTERVAL_S, "loops": N_FRAMES},
    stage_positions=positions,
    channels=IMAGING_CHANNELS,
    stim_channels=[STIM_CHANNEL],
    stim_frames=STIM_FRAMES,
    ref_channels=[REF_CHANNEL],
    ref_frames=[-1],
)

events = list(seq)
print(f"{len(events)} events")

## 6. Run the experiment

The stage returns to the first position at the end, so you can run the
notebook again from the same place.

In [ ]:
from faro.core.controller import Controller

ctrl = Controller(mic, pipeline, writer=writer)
assert ctrl.validate_events(events), "fix the warnings above before running"
ctrl.run_experiment(events).wait()
ctrl.finish_experiment()
mic.mmc.setXYPosition(x0, y0)
print("Experiment complete.")

## 7. Inspect the plate store

- The root holds the **plate** metadata (rows, columns, wells).
- Each position is a **well**: `A/1/0/`, `A/2/0/`, ...
- Each well has its own raw array `(t, c, y, x)`, without a position
  axis, and its own labels.

In [ ]:
import json

import zarr

store = zarr.open(writer._zarr_path, mode="r")
print(store.tree())

root_attrs = store.attrs.asdict()
ome = root_attrs.get("ome", root_attrs)
print(json.dumps(ome.get("plate", "no plate metadata"), indent=2))

In [ ]:
for col in range(1, N_POSITIONS + 1):
    well = store[f"A/{col}/0"]
    raw = well["0"]
    labels = well["labels"].attrs.get("labels", []) if "labels" in well else []
    print(f"Well A/{col}: raw shape={raw.shape}, labels={labels}")

In [ ]:
import pandas as pd

for fov in range(N_POSITIONS):
    tracks = pd.read_parquet(os.path.join(path, "tracks", f"{fov}_latest.parquet"))
    print(
        f"Position {fov}: {tracks['particle'].nunique()} cells "
        f"across {tracks['timestep'].nunique()} frames"
    )
tracks.head()

## 8. Open in napari

Drag `acquisition.ome.zarr` into napari. napari-ome-zarr shows the raw
channels of all wells as a mosaic. The label layers are in the store
but the plate reader does not load them; add one by hand:

```python
viewer.add_labels(store["A/1/0/labels/labels/0"][:], name="labels (well A/1)")
```

In [ ]:
print(f"Open in napari: {writer._zarr_path}")